<a href="https://colab.research.google.com/github/hagopjay/x3d.lab/blob/main/notebooks/06_Foundry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# x3d.lab · 06 Foundry

**Model-written scenes in, tests for the engines that disagree about them out.** The hog (05) runs one fixed corpus through every engine once. The Foundry runs in **rounds**, and every round starts from what the earlier ones learned:

| Step | What happens | Who it helps |
|---|---|---|
| **target** | picks nodes nobody knows much about yet: never tried, new in 4.1, not yet bridged in x3d.loom, failed by the models, or disputed last round | all |
| **generate** | NVIDIA models (or Claude, OpenAI, Gemini) write one scene per target node; x3d_mcp's exact fixes, then one repair turn | the models |
| **referee** | three judges that share no code: **x3d_mcp** (the fork), **lxml + the version's XSD**, and the **X3D object model** (UOM) where the XSD is known to be stricter | x3d_mcp, the XSDs |
| **engines** | **X_ITE** (the reference), the **x3d.loom** Babylon bridge (picture against X_ITE's picture), **Blender** (import, X3D round trip, is the export valid?) and **glTF** (Blender's export, the Khronos validator, X_ITE draws the glTF under the original camera) | X_ITE, x3d.loom, x3d-blender-io |
| **claims** | every disagreement becomes a claim that can be checked again: *"x3d_mcp rejects a scene the XSD accepts"*, *"the bridge draws the right shapes in the wrong colours"* | |
| **shrink** | each new claim is cut down to the smallest scene for which it is still true (delta debugging, re-checking with the engine involved) | whoever fixes it |
| **write** | **x3d.loom fixtures** pinned to X_ITE's own pixels, **x3d_mcp tests** (strict xfail), **issue drafts**, all in one kit zip | |
| **learn** | each model's mistakes become rules in its next system prompt; the scoreboard shows whether they helped | the next round |

**Runtime: economy first.** Nothing here needs an A100: the models run at NVIDIA, the validators and Blender's import/export run on the CPU, Chrome uses whatever GPU there is. A **CPU runtime** or a **T4** is enough; keep the A100 for the Cycles numbers in 02/05. A round of 8 nodes × 2 models takes about 10–20 minutes, most of it the bridge runner and shrinking.

**Keys:** in Colab **Secrets** (the key icon on the left) as `NVIDIA_API_KEY` (and optionally `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GEMINI_API_KEY`). Never paste a key into a cell. The `corpus` contender needs no key: it answers with x3d.loom's own fixtures (some with a planted defect), so you can watch a whole round for free.

## 1 · Start the lab

In [ ]:
#@title 1 · Start x3d_mcp and the dashboard { display-mode: "form" }
LOG_LEVEL = "WARN"  #@param ["DEBUG", "INFO", "WARN", "ERROR"]
LAB_REPO = "hagopjay/x3d.lab"
LAB_REF = "main"

import importlib.util, os, subprocess, sys
if not os.environ.get("X3DLAB_DEV") and (importlib.util.find_spec("x3dlab") is None or os.environ.get("X3DLAB_REINSTALL")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "x3dlab[notebook,tunnels,research] @ git+https://github.com/%s@%s" % (LAB_REPO, LAB_REF)], check=True)
import random
import x3dlab
from x3dlab import foundry, hog, keys

lab = x3dlab.up(tunnel="none", log_level=LOG_LEVEL)
print(lab.runtime_info["line"])

## 2 · API keys

**Load from Colab Secrets**, then **Test keys & list models**. A bare `nvidia` contender uses the model picked in the dropdown; `nvidia:<org/model>` names one exactly. The free NVIDIA tier allows about 40 requests a minute and the lab paces itself under that.

In [ ]:
#@title 2 · API keys (NVIDIA, Claude, OpenAI, Gemini)
keys.widget()

## 3 · Blender (for the Blender and glTF engines)

Only import, export and conversion run here: no Cycles unless you tick `CYCLES` in step 4, so this costs CPU seconds, not GPU minutes. Skip this cell and untick `BLENDER` and `GLTF` to leave Blender out.

In [ ]:
#@title 3 · Set up Blender { display-mode: "form" }
BLENDER_SOURCE = "auto"  #@param ["auto", "wheel", "tarball"]
bl = lab.setup_blender(BLENDER_SOURCE)
print("Blender %s · Cycles backend if asked for: %s" % (bl.version, bl.best_backend))

## 4 · The round

| Setting | What it changes |
|---|---|
| `CONTENDER_*` | `nvidia`, `nvidia:<org/model>`, `anthropic`, `openai`, `gemini`, or `corpus` (free, offline). Empty = skip |
| `TARGETS` / `NODES` | how many nodes to aim at, or a comma list of exact nodes (then `TARGETS` is ignored) |
| `REPAIR_ROUNDS` | model repair turns after x3d_mcp's exact fixes. The referee judges the **draft** too, so 0 still finds validator disagreements |
| engines | `XITE` is the reference the others are compared with; `LOOM` runs the bridge's own test runner; `BLENDER` and `GLTF` need step 3 |
| `MAX_SHRINKS`, `SHRINK_BUDGET` | how many new claims to shrink, and how many re-checks each may use. Validator claims re-check in milliseconds; bridge and glTF claims in seconds |
| `MAX_REQUESTS` | the round refuses to start if the worst case needs more model requests than this |
| `STATE` | where the memory between rounds lives. Point it at Google Drive to carry rules and known claims across sessions |

In [ ]:
#@title 4 · Settings { display-mode: "form" }
CONTENDER_1 = "nvidia"  #@param {type:"string"}
CONTENDER_2 = "nvidia:meta/llama-3.3-70b-instruct"  #@param {type:"string"}
CONTENDER_3 = ""  #@param {type:"string"}
TARGETS = 8  #@param {type:"slider", min:1, max:24, step:1}
NODES = ""  #@param {type:"string"}
X3D_VERSION = "4.0"  #@param ["4.0", "4.1"]
REPAIR_ROUNDS = 1  #@param {type:"slider", min:0, max:3, step:1}
XITE = True  #@param {type:"boolean"}
LOOM = True  #@param {type:"boolean"}
BLENDER = True  #@param {type:"boolean"}
GLTF = True  #@param {type:"boolean"}
CYCLES = False  #@param {type:"boolean"}
MAX_SHRINKS = 6  #@param {type:"slider", min:0, max:20, step:1}
SHRINK_BUDGET = 40  #@param {type:"slider", min:10, max:150, step:10}
MAX_REQUESTS = 80  #@param {type:"integer"}
CHROME_GPU = True  #@param {type:"boolean"}
STATE = ""  #@param {type:"string"}

engines = [e for e, on in (("xite", XITE), ("loom", LOOM), ("blender", BLENDER), ("gltf", GLTF)) if on]
cfg = foundry.FoundryConfig(contenders=[c.strip() for c in (CONTENDER_1, CONTENDER_2, CONTENDER_3) if c.strip()],
                            targets=TARGETS, nodes=[n.strip() for n in NODES.split(",") if n.strip()], version=X3D_VERSION,
                            repair_rounds=REPAIR_ROUNDS, engines=engines, cycles=CYCLES, max_shrinks=MAX_SHRINKS,
                            shrink_budget=SHRINK_BUDGET, max_requests=MAX_REQUESTS, chrome_gpu=CHROME_GPU, state=STATE)
if ("blender" in engines or "gltf" in engines) and lab.blender is None:
    print("⚠️  BLENDER/GLTF need step 3; they are left out for now.")
    cfg.engines = [e for e in engines if e not in ("blender", "gltf")]
from x3dlab import duel
ready, skipped = duel.resolve(duel.DuelConfig(contenders=cfg.contenders))
cfg.contenders = [c.spec for c in ready]
for c in ready:
    print("  ✓ %s" % c.label)
for spec, why in skipped.items():
    print("  ✗ %s: %s" % (spec, why))
est = foundry.estimate(cfg)
print("model requests: up to %d (limit %d)%s" % (est["requests_worst"], cfg.max_requests, "" if est["within_budget"] else "  ← above the limit"))

st = foundry.load_state(foundry.state_path(cfg))
n_next = len(st["rounds"]) + 1
loom_dir = hog.loom_checkout(cfg.loom_ref, cfg.loom_dir)
targets = foundry.choose_targets(cfg, st, foundry.loom_status(loom_dir), random.Random(cfg.seed or n_next))
print("\nround %d will aim at:" % n_next)
for t in targets:
    print("  %-28s %s  %-5s %s" % (t["node"], t.get("component", ""), t["version"], t["why"]))
if st["rules"]:
    print("\nrules carried over:", {k: len(v) for k, v in st["rules"].items()})

## 5 · Run rounds

Each round writes its report, card and kit as it goes (a stopped round still leaves files). Run this cell again for the next round: the targets move on, the rules carry over, and claims already reported with their smallest scene are only counted.

In [ ]:
#@title 5 · Run { display-mode: "form" }
ROUNDS = 1  #@param {type:"slider", min:1, max:5, step:1}
for i in range(ROUNDS):
    rnd = foundry.run(lab, cfg)
    foundry.show(rnd, lab)

**Reading a round.**

- **Scoreboard:** per contender, how many drafts were valid as written, after fixes, by the XSD; whether the model actually **used the node** it was asked for (a valid scene without it tests nothing); **sound** scenes are the ones every engine is expected to handle. **Δ vs last** is the draft-valid rate against the previous round of the same contender: the rules at work.
- **Claims:** **NEW** ones are shrunk and written up; **known** ones were reported in an earlier round (only counted now); **expected** ones are explained by a known limit (glTF has no text; the bridge has not started a node; Blender's importer skips a node) and are listed so you can see they were noticed; **same as above** shrank to the same scene as another claim: one problem seen from several nodes.
- **Gallery:** X_ITE · the bridge · the glTF trip drawn by X_ITE, all on the magenta key background, so a missing shape or a wrong colour is plain at a glance. The verdicts use silhouettes that forgive 2 px (lines and edges are never pixel-identical), and colour only when the shapes agree.
- **Rules for the next round:** taken only from real mistakes. When the XSD and the object model accept a draft the fork rejected, the fork was wrong and nothing is learned from it.

## 6 · The kit

Everything written for the projects, laid out like the repositories:

- `x3d.loom/x3d.webgpu/bridge/fixtures/foundry/*.x3d|.test.json`: bridge fixtures. Their pixels are X_ITE's, taken at flat spots (no edges) and kept only after a second run showed X_ITE reproduces them. `foundry.status` is **agrees** (a new regression test) or **ours-fails** (a target for the bridge). The bridge's default run reads `fixtures/*.test.json` only, so these never turn its CI red; run one with `node tools/run-tests.js --babylon <babylon.js> foundry/<Name>`.
- `x3d_mcp/tests/test_foundry_r<N>_*.py`: one strict-xfail test per validator claim, checked here against the fork's own `validate_xml` (a case that does not reproduce there is dropped). It passes while the bug is there; the day it is fixed it fails as XPASS: drop the marker and keep the test.
- `issues/*.md`: one draft per new claim with the smallest scene, every judge's word, and how it was found. Read before filing: a claim says engines disagree; the one in the title is the one the others point at.

In [ ]:
#@title 6 · Show the issue drafts and the kit { display-mode: "form" }
from x3dlab import ui
for f in sorted((rnd.dir / "kit" / "issues").glob("*.md")):
    ui.show("<details><summary style='font:14px system-ui'><b>%s</b></summary><pre style='white-space:pre-wrap;font-size:12px'>%s</pre></details>"
            % (ui.esc(f.name), ui.esc(f.read_text())))
for a in rnd.artifacts:
    print("%-14s %s %s" % (a["kind"], a.get("path") or "", a.get("status") or a.get("note") or ""))
print("kit zip:", rnd.dir / rnd.env["kit"] if rnd.env.get("kit") else "nothing written this round")

## 7 · Keep everything

Zips the last round's folder (report, card, scenes, pictures, kit) and the state file, then stops the lab. To chain rounds across Colab sessions, keep `state.json` (or set `STATE` in step 4 to a path on Google Drive).

In [ ]:
#@title 7 · Zip and stop { display-mode: "form" }
import shutil
sp = foundry.state_path(cfg)
shutil.copy(sp, rnd.dir / "state.json")
z = shutil.make_archive(str(rnd.dir), "zip", rnd.dir)
print("zip:", z, "(%.1f MB)" % (os.path.getsize(z) / 1e6))
try:
    from google.colab import files
    files.download(z)
except Exception:
    pass
lab.down()